In [0]:
orders = spark.read.format("delta") \
    .load("/Volumes/workspace/default/olist/silver/orders")

payments = spark.read.format("delta") \
    .load("/Volumes/workspace/default/olist/silver/payments")

items = spark.read.format("delta") \
    .load("/Volumes/workspace/default/olist/silver/order_items")

products = spark.read.format("delta") \
    .load("/Volumes/workspace/default/olist/silver/products")

customers = spark.read.format("delta") \
    .load("/Volumes/workspace/default/olist/silver/customers")

In [0]:
fact_sales = orders \
    .join(items, "order_id") \
    .join(payments, "order_id") \
    .join(customers, "customer_id") \
    .join(products, "product_id")

In [0]:
from pyspark.sql.functions import year, month, sum

In [0]:
monthly_sales = fact_sales.groupBy(
    year("order_purchase_timestamp").alias("year"),
    month("order_purchase_timestamp").alias("month")
).agg(
    sum("payment_value").alias("total_revenue")
)

In [0]:
display(
    dbutils.fs.ls("/Volumes/workspace/default/olist/silver")
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/olist/silver/customers/,customers/,0,1778398505607
dbfs:/Volumes/workspace/default/olist/silver/order_items/,order_items/,0,1778398505607
dbfs:/Volumes/workspace/default/olist/silver/orders/,orders/,0,1778398505607
dbfs:/Volumes/workspace/default/olist/silver/payments/,payments/,0,1778398505607
dbfs:/Volumes/workspace/default/olist/silver/products/,products/,0,1778398505607
dbfs:/Volumes/workspace/default/olist/silver/reviews/,reviews/,0,1778398505607


In [0]:
items = spark.read.format("delta") \
    .load("/Volumes/workspace/default/olist/silver/order_items")

In [0]:
display(monthly_sales)

year,month,total_revenue
2018,1,1408365.65000001
2018,4,1496811.5199999886
2017,4,505665.5300000008
2018,3,1475599.9499999913
2017,10,1021169.2699999986
2017,11,1583869.0100000002
2017,1,187779.4099999994
2016,10,73914.57999999996
2017,12,1042855.8599999979
2016,9,347.52


In [0]:
monthly_sales.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/olist/gold/monthly_sales")

In [0]:
top_categories = fact_sales.groupBy(
    "product_category_name"
).agg(
    sum("payment_value").alias("revenue")
).orderBy(
    "revenue",
    ascending=False
)

In [0]:
display(top_categories)

product_category_name,revenue
cama_mesa_banho,1712553.669999998
beleza_saude,1657373.1200000104
informatica_acessorios,1585330.4499999997
moveis_decoracao,1430176.3900000048
relogios_presentes,1429216.680000007
esporte_lazer,1392127.559999998
utilidades_domesticas,1094758.1300000006
automotivo,852294.330000002
ferramentas_jardim,838280.7500000013
cool_stuff,779697.9999999979


In [0]:
state_sales = fact_sales.groupBy(
    "customer_state"
).agg(
    sum("payment_value").alias("revenue")
).orderBy(
    "revenue",
    ascending=False
)

In [0]:
display(state_sales)

customer_state,revenue
SP,7597209.659999877
RJ,2769347.4399999706
MG,2326151.6399999945
RS,1147276.9999999984
PR,1064603.9899999956
BA,797410.3600000017
SC,786343.7100000014
GO,513879.00000000076
DF,432623.73000000016
ES,405805.34000000067


In [0]:
top_categories.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/olist/gold/top_categories")

state_sales.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/olist/gold/state_sales")